In [1]:
import pandas as pd
import numpy as np
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /home/dhanvi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/dhanvi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/dhanvi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

Load Dataset

In [2]:
df = pd.read_csv("data.csv")
df.head()

,source_txt,plagiarism_txt,label
0,A person on a horse jumps over a broken down a...,"A person is at a diner, ordering an omelette.",0
1,A person on a horse jumps over a broken down a...,"A person is outdoors, on a horse.",1
2,Children smiling and waving at camera,There are children present,1
3,Children smiling and waving at camera,The kids are frowning,0
4,A boy is jumping on skateboard in the middle o...,The boy skates down the sidewalk.,0


In [3]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367373 entries, 0 to 367372
Data columns (total 3 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   source_txt      367373 non-null  object
 1   plagiarism_txt  367369 non-null  object
 2   label           367373 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 8.4+ MB


(367373, 3)

0. handle none values 
1. lowercase
2. punctuation
3. numbers
4. extra spaces

In [4]:
import re
import string

def basic_clean(text):
    if pd.isna(text):   # handle NaN values
        return ""
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [5]:
df.isna().sum()

source_txt        0
plagiarism_txt    4
label             0
dtype: int64

In [6]:
df["source_clean"] = df["source_txt"].apply(basic_clean)
df["plagiarism_clean"] = df["plagiarism_txt"].apply(basic_clean)

In [7]:
df[["source_clean","plagiarism_clean"]].head()

,source_clean,plagiarism_clean
0,a person on a horse jumps over a broken down a...,a person is at a diner ordering an omelette
1,a person on a horse jumps over a broken down a...,a person is outdoors on a horse
2,children smiling and waving at camera,there are children present
3,children smiling and waving at camera,the kids are frowning
4,a boy is jumping on skateboard in the middle o...,the boy skates down the sidewalk


Tokenization

In [8]:
df["source_tokens"] = df["source_clean"].apply(word_tokenize)
df["plagiarism_tokens"] = df["plagiarism_clean"].apply(word_tokenize)

In [9]:
df[["source_tokens","plagiarism_tokens"]].head()

,source_tokens,plagiarism_tokens
0,"[a, person, on, a, horse, jumps, over, a, brok...","[a, person, is, at, a, diner, ordering, an, om..."
1,"[a, person, on, a, horse, jumps, over, a, brok...","[a, person, is, outdoors, on, a, horse]"
2,"[children, smiling, and, waving, at, camera]","[there, are, children, present]"
3,"[children, smiling, and, waving, at, camera]","[the, kids, are, frowning]"
4,"[a, boy, is, jumping, on, skateboard, in, the,...","[the, boy, skates, down, the, sidewalk]"


Stopword Removal

In [10]:
stop_words = set(stopwords.words("english"))

In [11]:
def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

In [12]:
df["source_tokens"] = df["source_tokens"].apply(remove_stopwords)
df["plagiarism_tokens"] = df["plagiarism_tokens"].apply(remove_stopwords)

In [13]:
df[["source_tokens","plagiarism_tokens"]].head()

,source_tokens,plagiarism_tokens
0,"[person, horse, jumps, broken, airplane]","[person, diner, ordering, omelette]"
1,"[person, horse, jumps, broken, airplane]","[person, outdoors, horse]"
2,"[children, smiling, waving, camera]","[children, present]"
3,"[children, smiling, waving, camera]","[kids, frowning]"
4,"[boy, jumping, skateboard, middle, red, bridge]","[boy, skates, sidewalk]"


Lemmatization

In [14]:
lemmatizer = WordNetLemmatizer()
def lemmatize(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

In [15]:
df["source_tokens"] = df["source_tokens"].apply(lemmatize)
df["plagiarism_tokens"] = df["plagiarism_tokens"].apply(lemmatize)

In [16]:
df[["source_tokens","plagiarism_tokens"]].head()

,source_tokens,plagiarism_tokens
0,"[person, horse, jump, broken, airplane]","[person, diner, ordering, omelette]"
1,"[person, horse, jump, broken, airplane]","[person, outdoors, horse]"
2,"[child, smiling, waving, camera]","[child, present]"
3,"[child, smiling, waving, camera]","[kid, frowning]"
4,"[boy, jumping, skateboard, middle, red, bridge]","[boy, skate, sidewalk]"


other way pos

In [17]:
import nltk
nltk.download("averaged_perceptron_tagger")

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/dhanvi/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [18]:
from nltk.corpus import wordnet

def get_wordnet_pos(tag):

    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [19]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):

    pos_tags = nltk.pos_tag(tokens)

    return [
        lemmatizer.lemmatize(word, get_wordnet_pos(pos))
        for word, pos in pos_tags
    ]

In [20]:
df["source_tokens"] = df["source_tokens"].apply(lemmatize_tokens)
df["plagiarism_tokens"] = df["plagiarism_tokens"].apply(lemmatize_tokens)

In [21]:
df[["source_tokens","plagiarism_tokens"]].head()

,source_tokens,plagiarism_tokens
0,"[person, horse, jump, break, airplane]","[person, diner, order, omelette]"
1,"[person, horse, jump, break, airplane]","[person, outdoors, horse]"
2,"[child, smile, wave, camera]","[child, present]"
3,"[child, smile, wave, camera]","[kid, frowning]"
4,"[boy, jump, skateboard, middle, red, bridge]","[boy, skate, sidewalk]"


Remove Short Words

In [22]:
def remove_short(tokens):
    return [word for word in tokens if len(word) > 2]

In [23]:
df["source_tokens"] = df["source_tokens"].apply(remove_short)
df["plagiarism_tokens"] = df["plagiarism_tokens"].apply(remove_short)

In [24]:
df[["source_tokens","plagiarism_tokens"]].head()

,source_tokens,plagiarism_tokens
0,"[person, horse, jump, break, airplane]","[person, diner, order, omelette]"
1,"[person, horse, jump, break, airplane]","[person, outdoors, horse]"
2,"[child, smile, wave, camera]","[child, present]"
3,"[child, smile, wave, camera]","[kid, frowning]"
4,"[boy, jump, skateboard, middle, red, bridge]","[boy, skate, sidewalk]"


Convert Tokens Back to Text

In [25]:
df["source_processed"] = df["source_tokens"].apply(lambda x: " ".join(x))
df["plagiarism_processed"] = df["plagiarism_tokens"].apply(lambda x: " ".join(x))

In [26]:
df[["source_processed","plagiarism_processed","label"]].head()

,source_processed,plagiarism_processed,label
0,person horse jump break airplane,person diner order omelette,0
1,person horse jump break airplane,person outdoors horse,1
2,child smile wave camera,child present,1
3,child smile wave camera,kid frowning,0
4,boy jump skateboard middle red bridge,boy skate sidewalk,0


In [27]:
df.to_csv("processed_dataset.csv", index=False)

TF-IDF Vectorization

In [28]:
# vectorizer = TfidfVectorizer()
vectorizer = TfidfVectorizer(ngram_range=(1,3), max_features=80000, min_df=2, max_df=0.9)
source_vectors = vectorizer.fit_transform(df["source_processed"])
plagiarism_vectors = vectorizer.transform(df["plagiarism_processed"])

In [29]:
print(source_vectors)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4958303 stored elements and shape (367373, 80000)>
  Coords	Values
  (0, 46888)	0.23161321728963716
  (0, 27496)	0.28429999804339506
  (0, 28814)	0.2359687233072469
  (0, 8536)	0.36855118163008466
  (0, 414)	0.39105298387507503
  (0, 46988)	0.5519970126839011
  (0, 27529)	0.4651821993794844
  (1, 46888)	0.23161321728963716
  (1, 27496)	0.28429999804339506
  (1, 28814)	0.2359687233072469
  (1, 8536)	0.36855118163008466
  (1, 414)	0.39105298387507503
  (1, 46988)	0.5519970126839011
  (1, 27529)	0.4651821993794844
  (2, 12271)	0.23713626831771986
  (2, 61826)	0.3091519532556064
  (2, 73934)	0.36881884093761685
  (2, 10403)	0.3168702825602422
  (2, 12658)	0.5379144116771876
  (2, 73946)	0.5678075663076356
  (3, 12271)	0.23713626831771986
  (3, 61826)	0.3091519532556064
  (3, 73934)	0.36881884093761685
  (3, 10403)	0.3168702825602422
  (3, 12658)	0.5379144116771876
  :	:
  (367369, 18169)	0.562684115435694
  (367370, 12271)	0.284

Features

Cosine Similarity

In [30]:
similarity_scores = []

for i in range(len(df)):

    sim = cosine_similarity(
        source_vectors[i],
        plagiarism_vectors[i]
    )[0][0]

    similarity_scores.append(sim)

In [31]:
df["cosine_similarity"] = similarity_scores

In [32]:
df[["source_processed","plagiarism_processed","cosine_similarity"]].head()

,source_processed,plagiarism_processed,cosine_similarity
0,person horse jump break airplane,person diner order omelette,0.081872
1,person horse jump break airplane,person outdoors horse,0.275777
2,child smile wave camera,child present,0.105611
3,child smile wave camera,kid frowning,0.000000
4,boy jump skateboard middle red bridge,boy skate sidewalk,0.037203


Word Overlap Feature

In [33]:
def word_overlap(row):

    s1 = set(row["source_processed"].split())
    s2 = set(row["plagiarism_processed"].split())

    overlap = len(s1.intersection(s2))

    total = len(s1.union(s2))

    return overlap / total

In [34]:
df["word_overlap"] = df.apply(word_overlap, axis=1)

In [35]:
df["len_source"] = df["source_processed"].apply(len)
df["len_plag"] = df["plagiarism_processed"].apply(len)
df["length_diff"] = abs(df["len_source"] - df["len_plag"])

Jaccard Similarity

In [36]:
def jaccard_similarity(row):

    s1 = set(row["source_processed"].split())
    s2 = set(row["plagiarism_processed"].split())

    return len(s1 & s2) / len(s1 | s2)

In [37]:
df["jaccard"] = df.apply(jaccard_similarity, axis=1)

Common Word Count

In [38]:
def common_words(row):

    s1 = set(row["source_processed"].split())
    s2 = set(row["plagiarism_processed"].split())

    return len(s1 & s2)

In [39]:
df["common_words"] = df.apply(common_words, axis=1)

Length Ratio

In [40]:
df["length_ratio"] = df["len_source"] / (df["len_plag"] + 1)

In [41]:
df

,source_txt,plagiarism_txt,label,source_clean,plagiarism_clean,source_tokens,plagiarism_tokens,source_processed,plagiarism_processed,cosine_similarity,word_overlap,len_source,len_plag,length_diff,jaccard,common_words,length_ratio
0,A person on a horse jumps over a broken down a...,"A person is at a diner, ordering an omelette.",0,a person on a horse jumps over a broken down a...,a person is at a diner ordering an omelette,"[person, horse, jump, break, airplane]","[person, diner, order, omelette]",person horse jump break airplane,person diner order omelette,0.081872,0.125000,32,27,5,0.125000,1,1.142857
1,A person on a horse jumps over a broken down a...,"A person is outdoors, on a horse.",1,a person on a horse jumps over a broken down a...,a person is outdoors on a horse,"[person, horse, jump, break, airplane]","[person, outdoors, horse]",person horse jump break airplane,person outdoors horse,0.275777,0.333333,32,21,11,0.333333,2,1.454545
2,Children smiling and waving at camera,There are children present,1,children smiling and waving at camera,there are children present,"[child, smile, wave, camera]","[child, present]",child smile wave camera,child present,0.105611,0.200000,23,13,10,0.200000,1,1.642857
3,Children smiling and waving at camera,The kids are frowning,0,children smiling and waving at camera,the kids are frowning,"[child, smile, wave, camera]","[kid, frowning]",child smile wave camera,kid frowning,0.000000,0.000000,23,12,11,0.000000,0,1.769231
4,A boy is jumping on skateboard in the middle o...,The boy skates down the sidewalk.,0,a boy is jumping on skateboard in the middle o...,the boy skates down the sidewalk,"[boy, jump, skateboard, middle, red, bridge]","[boy, skate, sidewalk]",boy jump skateboard middle red bridge,boy skate sidewalk,0.037203,0.125000,37,18,19,0.125000,1,1.947368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367368,A dog with a blue collar plays ball outside.,a dog is outside,1,a dog with a blue collar plays ball outside,a dog is outside,"[dog, blue, collar, play, ball, outside]","[dog, outside]",dog blue collar play ball outside,dog outside,0.110035,0.333333,33,11,22,0.333333,2,2.750000
367369,Four dirty and barefooted children.,four children have dirty feet.,1,four dirty and barefooted children,four children have dirty feet,"[four, dirty, barefooted, child]","[four, child, dirty, foot]",four dirty barefooted child,four child dirty foot,0.407809,0.600000,27,21,6,0.600000,3,1.227273
367370,Four dirty and barefooted children.,four kids won awards for 'cleanest feet',0,four dirty and barefooted children,four kids won awards for cleanest feet,"[four, dirty, barefooted, child]","[four, kid, award, clean, foot]",four dirty barefooted child,four kid award clean foot,0.114150,0.125000,27,25,2,0.125000,1,1.038462
367371,A man is surfing in a bodysuit in beautiful bl...,A man in a business suit is heading to a board...,0,a man is surfing in a bodysuit in beautiful bl...,a man in a business suit is heading to a board...,"[man, surf, bodysuit, beautiful, blue, water]","[man, business, suit, head, board, meeting]",man surf bodysuit beautiful blue water,man business suit head board meeting,0.010351,0.090909,38,36,2,0.090909,1,1.027027


Select features:

In [42]:
# X = df[[
#     "cosine_similarity",
#     "word_overlap",
#     "length_diff"
# ]]

# y = df["label"]

X = df[
[
"cosine_similarity",
"word_overlap",
"jaccard",
"common_words",
"length_diff",
"length_ratio"
]
]

y = df["label"]

Train Machine Learning Model

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Modeling

In [44]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [45]:
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)
pred_dt = dt_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, pred_dt))

Decision Tree Accuracy: 0.6335079959169786


In [46]:
model = LogisticRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)
accuracy_score(y_test, pred)

print("Logistic Regression:", accuracy_score(y_test, pred))

Logistic Regression: 0.6773732562095951


In [47]:
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
pred_nb = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, pred_nb))

Naive Bayes Accuracy: 0.6487512759441987


In [48]:
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)
pred_rf = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, pred_rf))

Random Forest Accuracy: 0.6661313371895202


Scale Features (Important for SVM)

In [49]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [50]:
svm_model = SVC(kernel="linear")
svm_model.fit(X_train_scaled, y_train)
pred_svm = svm_model.predict(X_test_scaled)

print("SVC Accuracy:", accuracy_score(y_test, pred_svm))

SVC Accuracy: 0.6772235454236135


In [51]:
import joblib

joblib.dump(svm_model, "plagiarism_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

Best Model

In [52]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred_svm))

              precision    recall  f1-score   support

           0       0.65      0.76      0.70     36795
           1       0.71      0.60      0.65     36680

    accuracy                           0.68     73475
   macro avg       0.68      0.68      0.67     73475
weighted avg       0.68      0.68      0.68     73475



In [53]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, pred_svm)

array([[27926,  8869],
       [14847, 21833]])

Detectionnn

In [55]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def detect_plagiarism(text1, text2):

    # preprocessing
    t1 = basic_clean(text1)
    t2 = basic_clean(text2)

    t1 = " ".join(remove_short(lemmatize(remove_stopwords(word_tokenize(t1)))))
    t2 = " ".join(remove_short(lemmatize(remove_stopwords(word_tokenize(t2)))))

    # TF-IDF similarity
    v1 = vectorizer.transform([t1])
    v2 = vectorizer.transform([t2])

    cos = cosine_similarity(v1, v2)[0][0]

    # word features
    s1 = set(t1.split())
    s2 = set(t2.split())

    common = len(s1 & s2)
    union = len(s1 | s2)

    overlap = common / union if union != 0 else 0
    jaccard = overlap

    len1 = len(t1.split())
    len2 = len(t2.split())

    length_diff = abs(len1 - len2)
    length_ratio = len1 / (len2 + 1)

    # features for SVM model
    features = pd.DataFrame(
        [[cos, overlap, jaccard, common, length_diff, length_ratio]],
        columns=[
            "cosine_similarity",
            "word_overlap",
            "jaccard",
            "common_words",
            "length_diff",
            "length_ratio"
        ]
    )

    features_scaled = scaler.transform(features)

    prediction = svm_model.predict(features_scaled)[0]

    # ⭐ NEW: semantic similarity
    emb1 = semantic_model.encode([text1])
    emb2 = semantic_model.encode([text2])

    semantic_score = cosine_similarity(emb1, emb2)[0][0]

    # final plagiarism score
    plagiarism_score = ((cos * 0.3) + (semantic_score * 0.5) + (overlap * 0.2)) * 100
    plagiarism_score = float(round(plagiarism_score, 2))

    if prediction == 1:
        result = "Plagiarism Detected"
    else:
        result = "Not Plagiarism"

    return {
        "plagiarism_percentage": plagiarism_score,
        "result": result
    }



In [ ]:
text1 = "Machine learning is a field of artificial intelligence"
text2 = "Machine learning is part of artificial intelligence"

detect_plagiarism(text1, text2)

NameError: name 'basic_clean' is not defined

In [ ]:
text1 = "Natural language processing enables computers to understand and analyze human language."
text2 = "NLP helps machines understand and process human language."

detect_plagiarism(text1, text2)

In [ ]:
text1 = "Deep learning uses neural networks with multiple layers"
text2 = "Deep learning is based on neural networks that contain many layers"

detect_plagiarism(text1, text2)

In [ ]:
text1 = "Python is widely used for machine learning and data science"
text2 = "Football is the most popular sport in the world"

detect_plagiarism(text1, text2)

In [ ]:
text1 = "Python is widely used for machine learning and data science"
text2 = "Football is the most popular sport in the world"

detect_plagiarism(text1, text2)

In [ ]:
text1 = "Big data technologies help organizations analyze massive datasets"
text2 = "Organizations analyze large datasets using big data technologies"

detect_plagiarism(text1, text2)

In [ ]:
# paraphrasing plagiarism

text1 = "Machines can improve their performance by learning from past information."
text2 = "Systems become better over time when they analyze previously collected data."

detect_plagiarism(text1, text2)

In [ ]:
text1 = """
Machine learning is a branch of artificial intelligence that focuses on building systems
that learn patterns from data. These systems improve automatically with experience
without being explicitly programmed.
"""

text2 = """
Machine learning is a part of artificial intelligence where systems learn patterns
from large amounts of data. These models can improve their performance over time
without direct programming.
"""

detect_plagiarism(text1, text2)

In [ ]:
text1 = """
Data science is a multidisciplinary field that uses statistics, programming,
and machine learning techniques to analyze large datasets and extract insights.
"""

text2 = """
Data science combines programming and statistical methods to examine data.
It helps organizations discover useful patterns and insights from large datasets.
"""

detect_plagiarism(text1, text2)